In [3]:
import os
import re
from bs4 import BeautifulSoup
import pandas as pd
import lxml

In [4]:
#посмотрим, какие символы используются в разметке
path = '/Users/air/Downloads/Ненецкие аудио/Siberian languages/XML файлы'

allchars = set()
for file in os.listdir(path):
    if file.endswith('.xml'):
        with open(os.path.join(path, file), encoding='utf-8') as f:
            content = f.read()
            matches = re.findall(r'<A>(.*?)</A>', content, flags=re.S) #нас интересует строка с разметкой, записанной между <A> и </A>
            for match in matches:
                allchars.update(c.lower() for c in match)

In [5]:
print(sorted(allchars)) #кириллицей записаны русские слова - их оставляем, скобки и знаки препинания почистим

['\n', ' ', '!', '(', ')', ',', '-', '.', '0', '1', '4', '5', '6', ':', '=', '?', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '«', '°', '´', '»', 'æ', 'í', 'ú', 'č', 'ŋ', 'š', 'ǝ', 'ə', 'ʹ', 'а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ы', 'ь', 'э', 'ю', 'я', 'ё', '“', '”', '′']


In [6]:
plain_vowels = {'a':'а','e':'э','o':'о','i':'ы','í':'ы','u':'у','ú':'у','ǝ':'а','ə':'а','æ':'э'} #не палатализованные
pal_vowels = {'a':'я','e':'е','o':'ё','i':'и','í':'и','u':'ю','ú':'ю','ǝ':'я','ə':'я','æ':'э'} #палатализованные
cons = {'p':'п','b':'б','m':'м','w':'в','n':'н','t':'т','d':'д','c':'ц','s':'с','l':'л','r':'р',
        'ŋ':'ӈ','k':'к','x':'х','g':'г','z':'з', 'š':'ш', 'č':'ч', 'f':'ф'}

vowels = set(plain_vowels) | set(pal_vowels) #соберём все гласные

In [7]:
def lattocyr(string):
    cyrword = []
    i = 0
    while i < len(string):
        char = string[i]
        if char in ('°', ':'): #выпускаем показатели краткости и долготы
            i += 1
            continue
        if char == 'y': #й и йотированные гласные
            if i+1 < len(string) and string[i+1] in pal_vowels:
                vowel = pal_vowels[string[i+1]]
                cyrword.append(vowel)
                i += 2
                continue
            cyrword.append('й')
            i += 1
            continue
        if char in cons:
            cyr = cons[char]
            if i+1 < len(string) and string[i+1] in ('ʹ', '′'): #палатализация
                if i+2 < len(string) and string[i+2] in vowels:
                    cyrword.append(cyr)
                    cyrword.append(pal_vowels.get(string[i+2], plain_vowels.get(string[i+2], string[i+2])))
                    i += 3
                    continue
                else:
                    cyrword.append(cyr + 'ь')
                    i += 2
                    continue
            cyrword.append(cyr)
            i += 1
            continue
        if char == 'q': #глухой тасер
            cyrword.append('"')
            i += 1
            continue
        if char == 'h': #звонкий тасер
            cyrword.append("'")
            i += 1
            continue
        if char in plain_vowels:
            cyrword.append(plain_vowels[char])
            i += 1
            continue
        cyrword.append(char)
        i += 1
    #print(cyrword)
    return ''.join(cyrword)
        

In [44]:
t = 'Mən′° s′íqw°mt′em°h, mən′°_d′i s′íqw°mt′em°h, mən′° il°nan° təmna s′íqw° ... mət°q ædakeda' #посмотрим на транслитерацию строки
print([lattocyr(i) for i in t.lower().split(' ')])

['мань', 'си"вмтем\',', 'мань_ди', 'си"вмтем\',', 'мань', 'ылнан', 'тамна', 'си"в', '...', 'мат"', 'эдакэда']


In [51]:
def extract_data_from_xml(file):
    with open(os.path.join(path, file), 'r', encoding='utf-8') as f:
        content = f.read()
    soup = BeautifulSoup(content, 'lxml-xml')
    audio_tags = soup.find_all('AUDIO')
    a_tags = soup.find_all('A')
    data = []
    for audio_tag, a_tag in zip(audio_tags, a_tags):
        start_time = audio_tag.get('start', '')
        end_time = audio_tag.get('end', '')
        text = a_tag.get_text(strip=True)
        text = re.sub(r'[.,?!()=_\[\]\«\»]', '', text)
        cyrillic_text = lattocyr(text.lower())
        data.append({
            'file_name' : os.path.join(path.replace('XML', 'wav'), file.replace('xml', 'wav')),
            'start_time': start_time,
            'end_time': end_time,
            'label': cyrillic_text,
            'length': float(end_time) - float(start_time)
        })
    return data

In [27]:
extract_data_from_xml('NENETS_01_23.xml')

[{'file_name': '/Users/air/Downloads/Ненецкие аудио/Siberian languages/wav файлы/NENETS_01_23.wav',
  'start_time': '0.0000',
  'end_time': '6.7429',
  'label': "семьянан мань аб яӈкням'",
  'length': 6.7429},
 {'file_name': '/Users/air/Downloads/Ненецкие аудио/Siberian languages/wav файлы/NENETS_01_23.wav',
  'start_time': '6.7429',
  'end_time': '10.8544',
  'label': 'няр няв войнана хааць',
  'length': 4.1115},
 {'file_name': '/Users/air/Downloads/Ненецкие аудио/Siberian languages/wav файлы/NENETS_01_23.wav',
  'start_time': '10.8544',
  'end_time': '14.1107',
  'label': 'войнана хаацьда больше ниць ту"',
  'length': 3.2562999999999995},
 {'file_name': '/Users/air/Downloads/Ненецкие аудио/Siberian languages/wav файлы/NENETS_01_23.wav',
  'start_time': '14.1107',
  'end_time': '21.4375',
  'label': 'один апой братув раненойӈэ раненойӈэ турӈась',
  'length': 7.3268},
 {'file_name': '/Users/air/Downloads/Ненецкие аудио/Siberian languages/wav файлы/NENETS_01_23.wav',
  'start_time'

In [52]:
all_data = []
for file in os.listdir(path):
    if file.endswith('.xml'):
        all_data.extend(extract_data_from_xml(file))

siberian_df = pd.DataFrame(all_data) #соберём всё в датафрейм

In [53]:
siberian_df[:10]

,file_name,start_time,end_time,label,length
0,/Users/air/Downloads/Ненецкие аудио/Siberian l...,9.133,14.600,ӈавна тасу' яв' вархана хэбидя я танявы,5.467
1,/Users/air/Downloads/Ненецкие аудио/Siberian l...,14.633,19.200,"тикы хэбидя яхна си""в хо вадёвы""",4.567
2,/Users/air/Downloads/Ненецкие аудио/Siberian l...,19.233,23.600,ӈобӈкуна ӈоб хасава ханеванць хая,4.367
3,/Users/air/Downloads/Ненецкие аудио/Siberian l...,23.633,30.533,"яля' ямпан' ядарӈа ӈамкэхартм хось я""ма ханеда...",6.900
4,/Users/air/Downloads/Ненецкие аудио/Siberian l...,30.567,34.933,"ихнянта ма “ха""манць хумпанци"" ядэрӈадм",4.366
5,/Users/air/Downloads/Ненецкие аудио/Siberian l...,34.967,36.567,мякни хэхадм,1.600
6,/Users/air/Downloads/Ненецкие аудио/Siberian l...,36.600,40.067,"ханедами яӈкуню""”",3.467
7,/Users/air/Downloads/Ненецкие аудио/Siberian l...,40.100,43.700,тад мякнта хая,3.600
8,/Users/air/Downloads/Ненецкие аудио/Siberian l...,43.733,52.333,"миӈа миӈа харта мята хось я""ма ёходакы",8.600
9,/Users/air/Downloads/Ненецкие аудио/Siberian l...,52.367,62.167,"харта ӈо"" нись харва"" хэбидя ян' тэвы""",9.800


In [54]:
#Postnasal obstruent weakening is reflected in standard orthography with 
# ‹мб›, ‹мд›, ‹мз›, ‹мг›, ‹нд›, ‹нз›, ‹ңг› for mp, mt, mc, mk, nt, nc, ŋk
replace_table = {'мп':'мб', 'мт':'мд', 'мц':'мз', 'мк':'мг', 'нт':'нд', 'нц':'нз', 'ӈк':'ӈг', '”': ''}
siberian_df['label'] = siberian_df['label'].str.replace(replace_table)

In [55]:
siberian_df[:10]

,file_name,start_time,end_time,label,length
0,/Users/air/Downloads/Ненецкие аудио/Siberian l...,9.133,14.600,ӈавна тасу' яв' вархана хэбидя я танявы,5.467
1,/Users/air/Downloads/Ненецкие аудио/Siberian l...,14.633,19.200,"тикы хэбидя яхна си""в хо вадёвы""",4.567
2,/Users/air/Downloads/Ненецкие аудио/Siberian l...,19.233,23.600,ӈобӈгуна ӈоб хасава ханеванзь хая,4.367
3,/Users/air/Downloads/Ненецкие аудио/Siberian l...,23.633,30.533,"яля' ямбан' ядарӈа ӈамгэхартм хось я""ма ханеда...",6.900
4,/Users/air/Downloads/Ненецкие аудио/Siberian l...,30.567,34.933,"ихнянда ма “ха""манзь хумбанзи"" ядэрӈадм",4.366
5,/Users/air/Downloads/Ненецкие аудио/Siberian l...,34.967,36.567,мякни хэхадм,1.600
6,/Users/air/Downloads/Ненецкие аудио/Siberian l...,36.600,40.067,"ханедами яӈгуню""",3.467
7,/Users/air/Downloads/Ненецкие аудио/Siberian l...,40.100,43.700,тад мякнда хая,3.600
8,/Users/air/Downloads/Ненецкие аудио/Siberian l...,43.733,52.333,"миӈа миӈа харта мята хось я""ма ёходакы",8.600
9,/Users/air/Downloads/Ненецкие аудио/Siberian l...,52.367,62.167,"харта ӈо"" нись харва"" хэбидя ян' тэвы""",9.800


In [56]:
#столько секунд придётся отбросить
print(siberian_df[siberian_df['length'] > 11]['length'].sum())
print(siberian_df[siberian_df['length'] < 3.0]['length'].sum())

523.1624000000002
297.6209000000003


In [57]:
siberian_df.to_csv('siberian.csv', index=False)